# Practice with PyPOTS, HuggingFace Datasets, and MAPIE

This notebook contains code for the experimentation with the pypots, datasets, and mapie Python libraries to help with the development of the project. It will consist of importing the data with datasets, constructing a DLinear model, and using MAPIE to create conformal prediction intervals. No forecasting.

In [1]:
# imports
import warnings
import torch
from pypots.imputation.dlinear import DLinear
from pypots.optim.adamw import AdamW
from uncertainty_aware_planner.data.data_utilities import (
    load_raw_data_for_imputation,
    split_by_window_imputation,
)
from uncertainty_aware_planner.evaluation.imputation_performance import evaluation_mnar
from uncertainty_aware_planner.utils.util_functions import set_seed

# notebook-level constants
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
CONFIG = {
    "saving_path": "./models/",
    "EPOCHS": 10,
    "batch_size": 128,
    "patience": 5,
    "d_model": 64,
    "lr": 0.001,
    "missing_rate": 0.3,
    "alpha": 1e-2,
}
set_seed(42)

warnings.simplefilter("ignore")
print(DEVICE)


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 

cuda


In [2]:
data = load_raw_data_for_imputation(missing_rate=CONFIG["missing_rate"])
windowed_data_view = split_by_window_imputation(data)
CONFIG.update(data["params"])

In [3]:
model = DLinear(
    n_steps=CONFIG["n_steps"],
    n_features=CONFIG["n_features"],
    moving_avg_window_size=CONFIG["patch_len"] // 2 * 2 + 1,
    individual=False,
    d_model=CONFIG["d_model"],
    ORT_weight=1,
    MIT_weight=1,
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["EPOCHS"],
    optimizer=AdamW(lr=CONFIG["lr"]),
    device=DEVICE,
    saving_path=CONFIG["saving_path"],
    patience=CONFIG["patience"],
)

2026-10-30 17:00:58 [INFO]: Using the given device: cuda
2026-10-30 17:01:00 [INFO]: Model files will be saved to ./models/20261030_T170100
2026-10-30 17:01:00 [INFO]: Tensorboard file will be saved to ./models/20261030_T170100/tensorboard
2026-10-30 17:01:00 [INFO]: Using customized MAE as the training loss function.
2026-10-30 17:01:00 [INFO]: Using customized MSE as the validation metric function.
2026-10-30 17:01:01 [INFO]: DLinear initialized with the given hyperparameters, the number of trainable parameters: 465,362


In [ ]:
# training a model on a particular window
model.fit(train_set=windowed_data_view[2]["train_set"])

2026-10-30 17:01:15 [INFO]: Epoch 001 - training loss (MAE): 0.1601
2026-10-30 17:01:30 [INFO]: Epoch 002 - training loss (MAE): 0.0655
2026-10-30 17:01:43 [INFO]: Epoch 003 - training loss (MAE): 0.0533
2026-10-30 17:01:55 [INFO]: Epoch 004 - training loss (MAE): 0.0471
2026-10-30 17:02:12 [INFO]: Epoch 005 - training loss (MAE): 0.0437
2026-10-30 17:02:32 [INFO]: Epoch 006 - training loss (MAE): 0.0413
2026-10-30 17:02:47 [INFO]: Epoch 007 - training loss (MAE): 0.0405
2026-10-30 17:02:59 [INFO]: Epoch 008 - training loss (MAE): 0.0391
2026-10-30 17:03:14 [INFO]: Epoch 009 - training loss (MAE): 0.0391
2026-10-30 17:03:27 [INFO]: Epoch 010 - training loss (MAE): 0.0388
2026-10-30 17:03:27 [INFO]: Finished training. The best model is from epoch#10.
2026-10-30 17:03:27 [INFO]: Saved the model to ./models/20261030_T170100/DLinear.pypots


In [5]:
# imputing data in the same window
results_train_set = model.predict(test_set=windowed_data_view[0]["train_set"])

In [6]:
eval_train_results = evaluation_mnar(
    input_data=windowed_data_view[0],
    imputation_data=results_train_set["imputation"],
    verbose=True,
)

wape 0.4887037821961686
wpe -0.3976044114841258
mase (lag=1, pooled scale) 1.1333875842475043
median per-window scaled error 0.795652455097336
evaluated days across all store-product series 237357 | days used for per-window diagnostic across all store-product series 235186


In [ ]:
results_test_set = model.predict(test_set=windowed_data_view[1]["train_set"])
eval_test_results = evaluation_mnar(
    input_data=windowed_data_view[1],
    imputation_data=results_test_set["imputation"],
    verbose=True,
)